# ISOM 835 · Explainer — Logistic Regression, Plain and Simple
**Suffolk University · Sawyer Business School · Fall 2026 · Prof. Hasan Arslan**

**After this notebook you can** explain to a manager what logistic regression does, use it to get a churn probability for any customer, and say what that probability means, without any formulas.

**How to use this notebook:** run the cells top to bottom (Shift + Enter). Read the short text above each cell first: it says **what the cell does** and **how to read the output**.

### Words to know

| Word | Plain English |
|---|---|
| **Target** | The yes/no thing we want to predict. Here: did the customer cancel (churn)? |
| **Feature** | A fact we know about the customer: months with us (tenure), contract type, monthly bill. |
| **Probability** | The model's answer: a number between 0 and 1. 0.40 = "40% chance this customer leaves." |
| **S-curve** | The shape logistic regression draws. It bends so the answer always stays between 0% and 100%. |
| **Cutoff** | The line that turns a probability into a yes/no decision. The default is 0.5. |

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

## 1. The business question

A phone and internet company (the Telco data from Session 1) wants to know **which customers are likely to cancel**, so it can reach out before they leave.

The next cell loads the data and shows the share of customers who churned.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

URL = 'https://raw.githubusercontent.com/harslan/isom-835/master/public/data/telco_churn.csv'
df = pd.read_csv(URL)
df['churned'] = (df['Churn'] == 'Yes').astype(int)     # 1 = cancelled, 0 = stayed

print(f'{len(df):,} customers')
print(f'{df["churned"].mean():.0%} of them churned')
df[['tenure', 'Contract', 'MonthlyCharges', 'Churn']].head()

## 2. Why not just draw a straight line?

Our target is only ever **0 (stayed)** or **1 (left)**. Let's try ordinary linear regression (a straight line) to predict churn from **tenure**, the number of months a customer has been with us.

**How to read the chart:** the dots are customers, sitting at 0 or 1. (We jitter them up and down a little so you can see them.) Watch where the red line goes at the edges.

In [ ]:
X1 = df[['tenure']]
y = df['churned']

line = LinearRegression().fit(X1, y)
months = pd.DataFrame({'tenure': np.arange(0, 121)})      # 0 to 10 years, to see where the line goes
line_pred = line.predict(months)

jitter = np.random.default_rng(835).uniform(-0.04, 0.04, len(df))
plt.figure(figsize=(8, 3.8))
plt.scatter(df['tenure'], y + jitter, s=4, alpha=0.15, color='gray')
plt.plot(months['tenure'], line_pred, color='#d64545', lw=2, label='straight line')
plt.axhline(0, color='black', lw=0.6)
plt.axhline(1, color='black', lw=0.6)
plt.xlabel('tenure (months)')
plt.ylabel('churned (0 = no, 1 = yes)')
plt.title('A straight line runs off the 0-to-1 scale')
plt.legend()
plt.show()

print(f'straight-line "probability" at 0 months:   {line_pred[0]:.2f}')
print(f'straight-line "probability" at 120 months: {line_pred[120]:.2f}  ← below zero: impossible for a probability')

A probability below 0% (or above 100%) makes no sense. We need a line that **bends**.

## 3. The S-curve: logistic regression

Logistic regression draws an **S-shaped curve** instead of a straight line. However far left or right you go, the answer stays between 0 and 1, so we can read it as a probability.

The next cell fits logistic regression with **one feature (tenure)** and draws its curve over the same data.

In [ ]:
logit1 = LogisticRegression().fit(X1, y)
curve = logit1.predict_proba(months)[:, 1]       # column 1 = probability of churn

plt.figure(figsize=(8, 3.8))
plt.scatter(df['tenure'], y + jitter, s=4, alpha=0.15, color='gray')
plt.plot(months['tenure'], line_pred, color='#d64545', lw=1.5, ls='--', label='straight line')
plt.plot(months['tenure'], curve, color='#2e8b57', lw=2.5, label='logistic S-curve')
plt.axhline(0, color='black', lw=0.6)
plt.axhline(1, color='black', lw=0.6)
plt.xlabel('tenure (months)')
plt.ylabel('chance of churning')
plt.title('Logistic regression bends and stays between 0 and 1')
plt.legend()
plt.show()

for m in [1, 12, 36, 72]:
    print(f'a customer with {m:2d} months of tenure: {curve[m]:.0%} chance of churning')

**Read it like a manager:** brand-new customers are the risky ones, and the risk falls steadily the longer someone stays. That's the whole idea of logistic regression: **facts about a customer go in, and a chance between 0% and 100% comes out.**

## 4. Use the model like a calculator

You can ask the model about **any** customer, even ones you make up. `predict_proba` returns the probability. The next cell asks about three made-up customers.

In [ ]:
new_customers = pd.DataFrame({'tenure': [2, 24, 60]})
new_customers['chance of churning'] = logit1.predict_proba(new_customers[['tenure']])[:, 1].round(2)
new_customers

## 5. Add a second fact: the contract type

Tenure is only part of the story. Contract type matters a lot: month-to-month customers can leave any time. Logistic regression handles several features at once.

`get_dummies` turns the text column `Contract` into 0/1 columns the model can use. The next cell fits the model on tenure **and** contract, then compares the three contract types for customers at the same tenure.

**How to read the chart:** each line is one contract type. Pick a tenure on the bottom axis and compare the three heights.

In [ ]:
X2 = pd.get_dummies(df[['tenure', 'Contract']], drop_first=True, dtype=int)
logit2 = LogisticRegression(max_iter=1000).fit(X2, y)
print('columns the model sees:', list(X2.columns))

plt.figure(figsize=(8, 3.8))
for contract, color in [('Month-to-month', '#d64545'), ('One year', '#f5a524'), ('Two year', '#2e8b57')]:
    grid = pd.DataFrame({'tenure': np.arange(0, 73),
                         'Contract_One year': int(contract == 'One year'),
                         'Contract_Two year': int(contract == 'Two year')})
    plt.plot(grid['tenure'], logit2.predict_proba(grid[X2.columns])[:, 1], color=color, lw=2.5, label=contract)
plt.xlabel('tenure (months)')
plt.ylabel('chance of churning')
plt.title('Same tenure, different contract, very different risk')
plt.legend()
plt.show()

example = pd.DataFrame({'tenure': [12, 12, 12],
                        'Contract_One year': [0, 1, 0],
                        'Contract_Two year': [0, 0, 1]})
chances = logit2.predict_proba(example[X2.columns])[:, 1]
for name, c in zip(['Month-to-month', 'One year', 'Two year'], chances):
    print(f'12 months of tenure, {name:15s} contract: {c:.0%} chance of churning')

> ### Going deeper (optional): what are the coefficients?
> Each feature gets one number, its **coefficient**. For tonight, only its **sign** matters:
> - **Positive** pushes the probability **up** (toward churn).
> - **Negative** pushes it **down** (toward staying).
>
> If you want one more step, `exp(coefficient)` turns it into an **odds multiplier**: the factor by which the odds of churning change. The next cell prints both, with one sentence in plain words.

In [ ]:
coefs = pd.Series(logit2.coef_[0], index=X2.columns)
summary = pd.DataFrame({'coefficient': coefs.round(3),
                        'direction': np.where(coefs > 0, 'raises churn risk', 'lowers churn risk'),
                        'odds multiplier': np.exp(coefs).round(2)})
print(summary, '\n')

per_year = np.exp(coefs['tenure'] * 12)
print(f'In words: each extra YEAR with the company multiplies the odds of churning by {per_year:.2f},')
print(f'that is, about {1 - per_year:.0%} lower odds per year (same contract type).')

## 6. From probability to decision

A probability is not an action yet. `predict()` turns probabilities into yes/no with a **cutoff of 0.5**: above 0.5 means "will churn." In Session 5 we learn that 0.5 is rarely the best cutoff for the business. For now, let's see what it does.

First, a surprise: with only tenure and contract, **no customer ever gets above 50%** (the riskiest, brand-new month-to-month customers sit right at about 50%). So `predict()` would say "nobody churns." The model isn't broken; the 0.5 line is just in the wrong place. Keep that in mind for Session 5.

For a fairer test, we give the model two more facts, **monthly bill** and **internet service**. We **train on 80%** of customers and **test on the other 20%**, which it has never seen. Then we count the four possible outcomes.

In [ ]:
print(f'highest churn chance the tenure + contract model gives anyone: {logit2.predict_proba(X2)[:, 1].max():.0%}\n')

X4 = pd.get_dummies(df[['tenure', 'Contract', 'MonthlyCharges', 'InternetService']], drop_first=True, dtype=int)
X_tr, X_te, y_tr, y_te = train_test_split(X4, y, test_size=0.2, stratify=y, random_state=835)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
pred = model.predict(X_te)                      # yes/no using the 0.5 cutoff

tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
print(f'Out of {len(y_te):,} test customers:')
print(f'  ✅ caught (predicted to leave, and did):        {tp}')
print(f'  ❌ missed (predicted to stay, but left):        {fn}')
print(f'  💸 false alarms (predicted to leave, stayed):   {fp}')
print(f'  👍 correctly left alone (predicted stay, did):  {tn}')
print()
print(f'model accuracy:                          {(tp + tn) / len(y_te):.1%}')
print(f'"nobody churns" guess (do-nothing baseline): {1 - y_te.mean():.1%}')

**Notice two things:**
1. The model beats the do-nothing guess, but only by a few points of accuracy. Accuracy is not the whole story.
2. Look at the **missed** count: at a 0.5 cutoff, the model still lets many leaving customers slip by. Choosing a better cutoff is the Session 5 topic.

## 7. Your turn

1. **Ask the calculator.** Use `logit1` (tenure only) to get the churn chance for customers with 6, 18, and 48 months of tenure.
   *Hint: copy the Step 4 cell and change the list of tenures.*
2. **Fill in the blanks.** Complete the code below to compare a **month-to-month** customer and a **two-year** customer, both with **3 months** of tenure. Which is riskier, and by how much?
3. **Explain it to your manager.** In two sentences with no code words: what does logistic regression give us, and why can't we use a straight line?

In [ ]:
# Your turn — work here
# Exercise 2: delete the # at the start of the four lines below, replace each ___ with a number, then run
# my_customers = pd.DataFrame({'tenure': [___, ___],
#                              'Contract_One year': [0, 0],
#                              'Contract_Two year': [___, ___]})    # 1 = has a two-year contract
# print(logit2.predict_proba(my_customers[X2.columns])[:, 1])

## What we learned

- **Logistic regression turns customer facts into a probability**, a chance between 0% and 100%.
- **A straight line fails** for yes/no targets because it runs below 0 and above 1; the **S-curve** never does.
- **More features, same idea:** contract type changes the risk a lot at the same tenure.
- **A coefficient's sign is its direction:** positive raises the risk, negative lowers it. Odds are optional detail.
- **A probability is not a decision.** `predict()` silently uses a 0.5 cutoff, and Session 5 shows how to choose a better one with dollars.

| Word | Plain English |
|---|---|
| **predict_proba** | "What's the chance?": returns probabilities |
| **predict** | "Yes or no?": applies the 0.5 cutoff |
| **Baseline** | The do-nothing guess you must beat, here "nobody churns" |
| **Train / test split** | Learn on 80%, grade on the unseen 20% |